In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


#  DeBERTa-v3  —  5-Fold CV Multiple-Choice fine-tuning
**AutoTokenizer + cosine LR scheduler + 5-model ensemble + W&B**

In [2]:
!pip install -q -U transformers datasets sentencepiece accelerate

import os, gc, numpy as np, pandas as pd, torch
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from transformers import (AutoTokenizer, AutoModelForMultipleChoice,
                          TrainingArguments, Trainer)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 105.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.7 MB/s eta 0:00:00


# Config


In [3]:
DATA    = "/kaggle/input/competitions/smart-mcq-solver-challenge"
MODEL   = "microsoft/deberta-v3-base"
OPTIONS = list("ABCDE")
MAXLEN  = 256
N_FOLDS = 5
EPOCHS  = 2
LR      = 2e-5
BATCH   = 4
SEED    = 42

# Load 

In [4]:
train = pd.read_csv(f"{DATA}/train.csv").reset_index(drop=True)
test  = pd.read_csv(f"{DATA}/test.csv").reset_index(drop=True)
train["label"] = train["answer"].map({c: i for i, c in enumerate(OPTIONS)})
test["label"]  = 0

# Tokenizer + preprocessing


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def preprocess(ex):
    first  = sum([[p] * 5 for p in ex["prompt"]], [])
    second = sum([[ex[o][i] for o in OPTIONS] for i in range(len(ex["prompt"]))], [])
    tok = tokenizer(first, second, truncation=True, max_length=MAXLEN)
    return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tok.items()}

drop_cols = ["id", "prompt", "A", "B", "C", "D", "E", "answer"]
full_ds = Dataset.from_pandas(train).map(preprocess, batched=True, remove_columns=drop_cols)

test_drop = [c for c in drop_cols if c in test.columns]
test_ds = Dataset.from_pandas(test).map(preprocess, batched=True, remove_columns=test_drop)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

# Collator

In [7]:
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    def __call__(self, features):
        name = "label" if "label" in features[0] else "labels"
        labels = [f.pop(name) for f in features]
        n, k = len(features), len(features[0]["input_ids"])
        flat = sum([[{key: f[key][i] for key in f} for i in range(k)] for f in features], [])
        batch = self.tokenizer.pad(flat, padding=self.padding,
                                   max_length=self.max_length, return_tensors="pt")
        batch = {key: v.view(n, k, -1) for key, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch

collator = DataCollatorForMultipleChoice(tokenizer)

# Metric

In [8]:
def map3(logits, labels):
    order = np.argsort(-logits, axis=1)
    s = 0.0
    for o, l in zip(order, labels):
        pos = int(np.where(o == l)[0][0])
        if pos < 3:
            s += 1.0 / (pos + 1)
    return s / len(labels)

def compute_metrics(p):
    logits, labels = p
    return {"map@3": map3(logits, labels),
            "acc": float((logits.argmax(1) == labels).mean())}

# W&B

In [9]:
import wandb
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
run = wandb.init(entity="ishankgpt02-na", project="23f1002033-t22026",
                 name="deberta-v3-5fold-cv",
                 config={"model": MODEL, "folds": N_FOLDS, "epochs": EPOCHS,
                         "lr": LR, "scheduler": "cosine", "max_len": MAXLEN})

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ishankgpt02 (ishankgpt02-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# 5-Fold CV


In [12]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_logits  = np.zeros((len(train), 5))
test_logits = np.zeros((len(test), 5))
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, train["label"])):
    print(f"FOLD {fold+1}/{N_FOLDS}")
    tr_ds = full_ds.select(tr_idx.tolist())
    va_ds = full_ds.select(va_idx.tolist())
    model = AutoModelForMultipleChoice.from_pretrained(MODEL)
    args = TrainingArguments(
        output_dir=f"out_fold{fold}",
        learning_rate=LR,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=8,
        num_train_epochs=EPOCHS,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        weight_decay=0.01,
        fp16=False,          
        bf16=True,           
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none",
        logging_steps=50,
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=tr_ds, eval_dataset=va_ds,
                      processing_class=tokenizer,
                      data_collator=collator,
                      compute_metrics=compute_metrics)
    trainer.train()
    oof_logits[va_idx] = trainer.predict(va_ds).predictions
    fold_map3 = map3(oof_logits[va_idx], train["label"].values[va_idx])
    fold_scores.append(fold_map3)
    print(f"Fold {fold+1} MAP@3: {fold_map3:.4f}")
    wandb.log({"fold": fold + 1, "fold_map@3": fold_map3})
    test_logits += trainer.predict(test_ds).predictions / N_FOLDS
    del model, trainer; gc.collect(); torch.cuda.empty_cache()

FOLD 1/5


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weigh

Epoch,Training Loss,Validation Loss,Map@3,Acc
1,0.000000,nan,0.384167,0.185000
2,0.000000,nan,0.384167,0.185000


Fold 1 MAP@3: 0.3842


...

FOLD 5/5


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weigh

Epoch,Training Loss,Validation Loss,Map@3,Acc
1,0.000000,nan,0.381667,0.182500
2,0.000000,nan,0.381667,0.182500


Fold 5 MAP@3: 0.3817


# Overall CV


In [13]:
cv_map3 = map3(oof_logits, train["label"].values)
print(f"OOF CV MAP@3: {cv_map3:.4f}")
print("per-fold:", [round(s, 4) for s in fold_scores])
wandb.log({"cv_map@3": cv_map3, "cv_std": float(np.std(fold_scores))})
run.summary["cv_map@3"] = cv_map3
run.finish()

OOF CV MAP@3: 0.3835
per-fold: [0.3842, 0.3842, 0.3842, 0.3833, 0.3817]


cv_map@3,▁
cv_std,▁
fold,▁▃▅▆█
fold_map@3,███▆▁
cv_map@3,0.3835
cv_std,0.00097
fold,5
fold_map@3,0.38167


# Submission (5-fold ensemble)


In [14]:
order = np.argsort(-test_logits, axis=1)
preds = [" ".join(OPTIONS[i] for i in row[:3]) for row in order]
submission = pd.DataFrame({"ID": test["id"], "Prediction": preds})
submission.to_csv("submission.csv", index=False)
print("submission.csv saved (5-fold ensemble)")

submission.csv saved (5-fold ensemble)
